# Megasweep PL Analysis

Refactored notebook — all heavy logic lives in `megasweep_analysis.py`.  
**Edit the CONFIG cell (Cell 2) then run all cells.**

Sections:
1. Setup
2. CONFIG ← edit here
3. Load CSV
4. Compute maps
5. Original-axis 2D maps
6. Transformed-axis 2D maps (Doping / Efield)
7. Line cuts
8. (Optional) Two-level fitting

## 1. Setup

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Make sure the app folder is on the path
APP_DIR = os.path.dirname(os.path.abspath('__file__'))
if APP_DIR not in sys.path:
    sys.path.insert(0, APP_DIR)

from megasweep_analysis import (
    load_megasweep_csv,
    compute_intensity_map,
    compute_peak_energy_map,
    build_grid,
    compute_transformed_coords,
    extract_line_cut,
    plot_map,
    plot_line_cut_spectrogram,
    save_figure,
    save_map_csv,
    save_line_csv,
    twolevel_fitting,
)
print('Imports OK')

## 2. CONFIG  ← edit this cell

In [ ]:
# ── Input ──────────────────────────────────────────────────────────────────
CSV_PATH = r'../megasweep/$YZD303$~$pen1$~$0T1.688K$~$PL660nm0.447uw1500msx1c932nm$~.csv'

# Axis columns — set to None to auto-detect from first two columns
X_COL = None   # e.g. 'Vbg'  or  None
Y_COL = None   # e.g. 'Vtg'  or  None

# TG / BG assignment
# TG_IS_Y = True  → TG = Y_COL (second column), BG = X_COL (first column)
TG_IS_Y = True

# ── Transformed-axis ratio ─────────────────────────────────────────────────
# Doping = RATIO * TG + BG
# Efield  = RATIO * TG - BG
RATIO = 0.9

# ── Intensity map settings ─────────────────────────────────────────────────
INT_MIN  = 1.25    # lower energy bound (eV)
INT_MAX  = 1.425   # upper energy bound (eV)
BASELINE = 595     # flat background per spectral channel

# ── Peak-energy map smoothing ──────────────────────────────────────────────
PEAK_SG_WINDOW = 25
PEAK_SG_POLY   = 1

# ── Output ────────────────────────────────────────────────────────────────
OUTPUT_DIR = '../megasweep_output'

## 3. Load CSV

In [ ]:
data = load_megasweep_csv(CSV_PATH, x_col=X_COL, y_col=Y_COL)

src    = data['source_name']
x_name = data['x_name']
y_name = data['y_name']

print(f'Source : {src}')
print(f'x-axis : {x_name}  ({len(np.unique(data["x_data"]))} unique values)')
print(f'y-axis : {y_name}  ({len(np.unique(data["y_data"]))} unique values)')
print(f'Spectra: {data["Intensity"].shape[0]} points × {data["Intensity"].shape[1]} channels')
print(f'Energy : {data["energy"].min():.3f} – {data["energy"].max():.3f} eV')

In [ ]:
# Preview a single spectrum (change row_index to inspect different points)
row_index = 0

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(data['energy'], data['Intensity'][row_index], lw=1.2)
ax.set_xlabel('Energy (eV)')
ax.set_ylabel('PL intensity (a.u.)')
ax.set_title(f'Spectrum at row {row_index}  '
             f'({x_name}={data["x_data"][row_index]:.2f}, '
             f'{y_name}={data["y_data"][row_index]:.2f})')
plt.tight_layout()
plt.show()

## 4. Compute maps

In [ ]:
print(f'Computing intensity map  [{INT_MIN}, {INT_MAX}] eV, baseline={BASELINE} ...')
intensity_flat = compute_intensity_map(data, INT_MIN, INT_MAX, BASELINE)

print(f'Computing peak-energy map  (SG window={PEAK_SG_WINDOW}) ...')
peak_e_flat = compute_peak_energy_map(data, PEAK_SG_WINDOW, PEAK_SG_POLY)

print('Building 2D grids ...')
X2D, Y2D, Z_intensity, idx_flat, n_x, n_y = build_grid(
    data['x_data'], data['y_data'], intensity_flat)

# peak energy shares the same grid
Z_peak_e = peak_e_flat[idx_flat].reshape(n_x, n_y)

print(f'Grid shape: {n_x} × {n_y}')

## 5. Original-axis 2D maps  (Vbg × Vtg)

In [ ]:
orig_dir = os.path.join(OUTPUT_DIR, 'original_axes_maps')
os.makedirs(orig_dir, exist_ok=True)

# — Intensity map —
fig, ax = plot_map(X2D, Y2D, Z_intensity,
                   x_label=f'{x_name} (V)', y_label=f'{y_name} (V)',
                   z_label='Integrated PL (a.u.)',
                   title=f'Intensity  |  {src}')
save_figure(fig, os.path.join(orig_dir, f'{src}_intensity_original.png'))
save_map_csv(X2D, Y2D, Z_intensity,
             os.path.join(orig_dir, f'{src}_intensity_original.csv'),
             x_name=x_name, y_name=y_name, z_name='intensity')
plt.show()

# — Peak energy map —
fig, ax = plot_map(X2D, Y2D, Z_peak_e,
                   x_label=f'{x_name} (V)', y_label=f'{y_name} (V)',
                   z_label='Peak energy (eV)',
                   title=f'Peak energy  |  {src}')
save_figure(fig, os.path.join(orig_dir, f'{src}_peak_energy_original.png'))
save_map_csv(X2D, Y2D, Z_peak_e,
             os.path.join(orig_dir, f'{src}_peak_energy_original.csv'),
             x_name=x_name, y_name=y_name, z_name='peak_energy_eV')
plt.show()

## 6. Transformed-axis maps  (Doping / Efield)

In [ ]:
trans_dir = os.path.join(OUTPUT_DIR, 'transformed_axes_maps')
os.makedirs(trans_dir, exist_ok=True)

Doping2D, Efield2D = compute_transformed_coords(X2D, Y2D, RATIO, tg_is_y=TG_IS_Y)

print(f'Doping = {RATIO}·TG + BG')
print(f'Efield  = {RATIO}·TG - BG')

for qty_name, Z2D, z_label in [
    ('intensity',   Z_intensity, 'Integrated PL (a.u.)'),
    ('peak_energy', Z_peak_e,    'Peak energy (eV)'),
]:
    fig, ax = plot_map(Doping2D, Efield2D, Z2D,
                       x_label='Doping (V)', y_label='Efield (V)',
                       z_label=z_label,
                       title=f'{qty_name} (transformed)  |  {src}')
    save_figure(fig, os.path.join(trans_dir, f'{src}_{qty_name}_transformed.png'))
    save_map_csv(Doping2D, Efield2D, Z2D,
                 os.path.join(trans_dir, f'{src}_{qty_name}_transformed.csv'),
                 x_name='Doping_V', y_name='Efield_V', z_name=qty_name)
    plt.show()

## 7. Line cuts

Edit `LINE_CUTS` below to add / change cuts.

In [ ]:
LINE_CUTS = [
    # {'cut_type': 'doping' or 'efield', 'c_value': float, 'epsilon': float}
    {'cut_type': 'doping', 'c_value':  0.0, 'epsilon': 0.03},
    {'cut_type': 'doping', 'c_value': -5.0, 'epsilon': 0.03},
    {'cut_type': 'efield', 'c_value':  0.0, 'epsilon': 0.03},
]

In [ ]:
lines_dir = os.path.join(OUTPUT_DIR, 'extracted_lines')
os.makedirs(lines_dir, exist_ok=True)

raw_data = data['raw_data']
x_data   = data['x_data']
y_data   = data['y_data']
energy   = data['energy']

for lc_spec in LINE_CUTS:
    cut_type = lc_spec['cut_type']
    c_value  = lc_spec['c_value']
    epsilon  = lc_spec.get('epsilon', 0.03)

    lc = extract_line_cut(
        raw_data, x_data, y_data, energy,
        cut_type=cut_type, c_value=c_value,
        ratio=RATIO, epsilon=epsilon,
        tg_is_y=TG_IS_Y,
    )

    if lc['spectra'].shape[0] == 0:
        print(f'WARNING: {cut_type}={c_value} → no points found, skipping')
        continue

    print(f'{cut_type} = {lc["c_value_used"]:.3f}  →  {lc["spectra"].shape[0]} points')

    c_str = f'{lc["c_value_used"]:.3f}'.replace('.', 'p').replace('-', 'm')
    stem  = f'{src}_{cut_type}{c_str}'

    fig, ax = plot_line_cut_spectrogram(lc)
    save_figure(fig, os.path.join(lines_dir, f'{stem}.png'))
    save_line_csv(lc, os.path.join(lines_dir, f'{stem}.csv'))
    plt.show()

## 8. (Optional) Two-level fitting

Fit a two-level anti-crossing model to a single peak-position dataset.

In [ ]:
# Example: fit the first line cut's peak energy vs. its varying axis
# Requires you to already have lc defined from Section 7.

# lc is the last line-cut dict from the loop above.
# If you want a specific one, call extract_line_cut again:
#
# lc = extract_line_cut(raw_data, x_data, y_data, energy,
#                        cut_type='doping', c_value=0.0, ratio=RATIO,
#                        epsilon=0.03, tg_is_y=TG_IS_Y)

# Get peak energy along the line
from megasweep_analysis import compute_peak_energy
peak_along_line = compute_peak_energy(lc['spectra'], lc['energy'],
                                       sg_window=PEAK_SG_WINDOW, sg_poly=PEAK_SG_POLY)
gate_axis = lc['axis_values']

fig, params, perr = twolevel_fitting(gate_axis, peak_along_line,
                                      title=f'2-level fit  |  {src}')
plt.show()
print('Params:', params)
print('Errors:', perr)